# 34 · Point data, checked against geopandas

## What this checks

**Point data** can read a table into points. It takes x from the column named as the longitude, and y from the
column named as the latitude. This check covers release 0.19.0, and only this table source.

The New York City worked example uses it to place six ozone monitors. Their daily readings are what the study
averages. A point put in the wrong place, or a row silently dropped, changes every result built on it.

## How it is checked

We make the same points with **geopandas**, a widely used Python library for map data. Its function for making points from x
and y is written by other authors, and shares no code with Fieldwork. The order of its arguments is worth checking: x
comes first, and x is the longitude.

Which rows get a point is Fieldwork's own rule, so we restate it here rather than borrow it. A row is not placed when
its position is empty, not a finite number, a longitude beyond 180 or a latitude beyond 90. geopandas would make a
point at a latitude of 90.0001. Fieldwork keeps such a row with no position and counts it.

The data are two tables. One is the six ozone monitors, totalled by site and position. The other is nine rows built
so that a run must report some of them rather than refuse them.

## What should happen

1. The same rows are placed and not placed, in the same order.
2. Every placed point has exactly the x and y that geopandas gives for the same row.
3. The number of distinct positions is the same on both sides.
4. geopandas reports the coordinate system EPSG:4326, with the longitude as x.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "shapely", "pyproj", "geopandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_34_points_from_xy import run

report = run(Path.cwd() / "fixtures" / "points-from-xy-001.json")
print("agrees:", report["agrees"])
for c in report["cases"]:
    print(f"{c['note']}: {c['placed']} of {c['rows']} placed; not placed {c['unplaced']['fieldwork']}; "
          f"positions {c['distinctPositions']}; {c['crs']}")

## The result on record

First recorded on 5 November 2026, against release 0.4.0. Rerun on 26 November 2026, against release 0.19.0:
**Agrees**.

- **The ozone monitors.** All 6 rows were placed, at 6 distinct positions. Every point is exactly the one geopandas
  makes.
- **The rows to report.** 5 of the 9 rows were placed, at 4 distinct positions. Every placed point is exactly the
  one geopandas makes.

Running the cell above repeats the check on the same data.

## What this does not show

- That a published position is where the monitor stands. That is the claim of whoever publishes the data.
- Any coordinate system other than longitude and latitude (CRS84).
- Point data's other sources: a file, GeoJSON, GeoPackage, or pins placed by hand. Only the table source is checked.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture:", report["fixture"]["id"], report["fixture"]["operation"]["widget"], report["fixture"]["operation"]["release"])